

# Notebook 



-----------

Before running this notebook, take a look at the _settings.py file. This python script defines paths to data files and directories used in this notebook.
It also defines some configurations, resolutions and options to use when running the script.
Adjust the paths as needed before proceeding with the rest of the notebook.

In [1]:
# Let's start by importing all necessary packages and functions defined in this folder.

import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
import pickle as pk
from scipy import interpolate
import regionmask
import glob, os, re, sys
import openpyxl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import warnings
from math import ceil 
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 20)
%matplotlib inline 

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sys.path.append('../..') # location of dem4cli package
#import demographics4climate as dem4cli # to call them as dem4cli.function 
from dem4cli import * # to call fxns directly


In [2]:
print(np.__version__)
print(pd.__version__)

1.25.1
2.0.3


In [3]:
# This code can be run using different configurations, different resolutions and different options. Which ones are we using?

flags

{'version': 2,
 'pop_resolution': 0.1,
 'GMT_mapping': 'year_to_year',
 'cohort_sizes_source': 'UNWPP2024',
 'countrymask': 'shapefile'}

In [4]:
# We'll run the analysis on a domain that can be specified here: Lat S, Lat N, Lon W, Lon E.

bbox_indiaws = [ 2.00, 40.00, 66.00, 100.00 ]
bbox_europe = [ 31.99,  71.09, -14.96,  34.94]
bbox_iberia = [ 36.00, 44.00, -10.00, 5.00]

## Population preprocessing

This section take care of loading and preprocessing all demographic data required for the analysis. 
It uses a wrapper function called preprocess_all_country_data and defined in population_demographics.py.

You can have a look at how preprocess_all_country_data is defined to understand which functions it calls and what it does. 
You can also read the Methods part of Grant et al. (2025) or the Supplementary Material of Thiery et al. (2021) to understand what demographic data are used in this analysis and how they are processed for this purpose.

In [5]:
# Make sure that the function below call the shapefile you have just created.
# Make sure that the path to data directories are correctly defined to be able to load the required data.

d_countries = preprocess_all_country_data(

    filepath_lifeexpectancy = filepath_lifeexpectancy, # life expectancy data
    start_birthyear=1950,
    end_birthyear=2025,                 # endyear is taken from end_birthyear + max life expectancy

    dir_cohortsizes = dir_cohortsizes,  # cohort size data
    data_source_cohorts='UNWPP2024',
    extend_method='linear',             # note, 'slinear' not implemented for UNWPP2024
    by_sex=False,                       # NOTE by_sex not implemented
                                            
    dir_population= dir_population,     # gridded pop data 
    ssp=2,
    urbanrural=False,                   # NOTE urbanrural not implemented for v2
    bbox = bbox_iberia,

    filepath_countrymask = filepath_countrymask,
    data_source_countrymask = 'shapefile',
    fillcoast=False, 
    fix_smallislands=False,
    
    filepath_world_bank = filepath_world_bank_meta, # metadata 
    filepath_lookuptable = filepath_lookuptable,    # country filtering
    filter_countries=True,
    worldbank_filter=True, 
    )

df_countries = d_countries['info_pop']
gdf_country_borders = d_countries['borders'] 
da_regions = df_countries['region'].unique()
da_population = d_countries['population_map']
df_birthyears = d_countries['birth_years'] # NA
df_life_expectancy_5 = d_countries['life_expectancy_5']
da_cohort_size = d_countries['cohort_size']
countries_regions, countries_mask = d_countries['mask']

load_country_metadata took 0.20 s
load_unwpp_lifeexpectancy took 23.43 s
get_life_expectancies took 0.00 s
loading cohort sizes from UNWPP2024
load_cohort_sizes took 26.50 s
interpolate_cohortsize_countries took 0.02 s
opening compass - historical
opening compass - ssp2
load_population took 3.24 s
load_countrymask took 1.98 s
preprocess_all_country_data took 55.41 s


In [6]:
df_life_expectancy_5

Country,Algeria,Andorra,France,Portugal,Spain
Year,,,,,
1950,60.0833,78.4736,77.6547,76.417,77.7527
1951,59.9968,78.7848,77.4317,75.5057,77.2738
1952,59.8907,79.0685,77.7348,76.3479,77.2174
1953,59.7686,79.449,78.7521,77.898,78.9503
1954,59.6571,79.7661,78.6738,77.5989,78.8173
...,...,...,...,...,...
2021,84.3678,90.9386,90.0047,89.1233,90.3076
2022,84.5161,91.0655,90.1282,89.2859,90.4504
2023,84.6631,91.1889,90.2457,89.4337,90.5779


In [15]:
print(df_life_expectancy_5.columns)
print(df_life_expectancy_5.index)

Index(['Algeria', 'Andorra', 'France', 'Portugal', 'Spain'], dtype='object', name='Country')
Index([1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961,
       1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973,
       1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985,
       1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997,
       1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009,
       2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021,
       2022, 2023, 2024, 2025],
      dtype='int64', name='Year')


In [20]:
print(df_life_expectancy_5.loc[1950, 'Portugal'])

76.417


## Land fraction exposed



In [8]:
# Load results from RIME-X

df_timeseries_quantiles = pd.read_csv('/data/brussel/vo/000/bvo00012/vsc11359/data_rime-x/CurPol_NZ2050_fwixd_HI-caution_PRT_pop.csv')
df_timeseries_quantiles

,quantile,year,fwixd,fwixd.1,HI-caution,HI-caution.1
0,NaN,NaN,NGFS CurPol,NGFS Net-Zero 2050,NGFS CurPol,NGFS Net-Zero 2050
1,0.50,1950.0,19.20358502035409,19.20358502035409,74.24875862768988,74.24875862768988
2,0.50,1951.0,19.20358502035409,19.20358502035409,74.24875862768988,74.24875862768988
3,0.50,1952.0,19.20358502035409,19.20358502035409,74.24875862768988,74.24875862768988
4,0.50,1953.0,19.20358502035409,19.20358502035409,74.24875862768988,74.24875862768988
...,...,...,...,...,...,...
1053,0.95,2096.0,61.832905719699994,38.76497034626531,131.62428748045113,95.26717711486944
1054,0.95,2097.0,62.04387501964119,38.70479678167208,131.99333044493972,95.0983857674809
1055,0.95,2098.0,62.24593717793041,38.58210380645113,132.25846071628794,94.92220041330002
1056,0.95,2099.0,62.62355088087419,38.507882775499304,132.6448320099581,94.7545405703605


In [10]:
columns = [
    ('quantile', ''),
    ('year', ''),
    ('fwixd', 'NGFS CurPol'),
    ('fwixd', 'NGFS Net-Zero 2050'),
    ('HI-caution', 'NGFS CurPol'),
    ('HI-caution', 'NGFS Net-Zero 2050')
]
df_timeseries_quantiles.columns = pd.MultiIndex.from_tuples(columns)

In [11]:
print(df_timeseries_quantiles.columns)

MultiIndex([(  'quantile',                   ''),
            (      'year',                   ''),
            (     'fwixd',        'NGFS CurPol'),
            (     'fwixd', 'NGFS Net-Zero 2050'),
            ('HI-caution',        'NGFS CurPol'),
            ('HI-caution', 'NGFS Net-Zero 2050')],
           )


In [ ]:
# Compute sum over a period of 70 years (proxy for lifetime exposure calculations)

start_birthyear = 1950
end_birthyear = 2025

## Process dataframe
# Ensure 'year' is numeric (if it's a string, convert it)
df_timeseries_quantiles['year'] = pd.to_numeric(df_timeseries_quantiles['year'], errors='coerce')

# Sort the DataFrame by 'year' for efficient slicing
df_timeseries_quantiles = df_timeseries_quantiles.sort_values('year')


## Load life expectancy data



# Initialize a dictionary to store results for each starting year
d_lifetime_exp = {}

# Iterate over each starting year from 1950 to 2025
for birth_yr in range(start_birthyear, end_birthyear+1):

    # Get lifetime expectancy for each birth cohort
    lifetime_expectancy = df_life_expectancy_5.loc[1950, 'Portugal']

    # Define death year based on life expectancy
    death_yr = birth_yr + np.floor(lifetime_expectancy)

    # Filter for years between yr and yr+ (inclusive)
    filtered_df = df_timeseries_quantiles[(df_timeseries_quantiles['year']>= birth_yr) & (df_timeseries_quantiles['year'] <= death_yr-1)]

    # Group by 'quantile' and sum the values
    summed_df = filtered_df.groupby('quantile').sum()

### Problem here
    summed_df = summed_df + (lifetime_expectancy - np.floor(lifetime_expectancy)) * df_timeseries_quantiles[df_timeseries_quantiles['year']== death_yr]

    # Store the result for this starting year
    d_lifetime_exp[birth_yr] = summed_df

# Combine all results into a single DataFrame (optional)
df_lifetime_exp = pd.concat(d_lifetime_exp, names=['birth year', 'quantile'])

TypeError: can't multiply sequence by non-int of type 'float'

In [22]:
df_lifetime_exp

year  \
                                
birth year quantile             
1950       0.05      151050.0   
           0.10      151050.0   
           0.25      151050.0   
           0.50      151050.0   
           0.75      151050.0   
...                       ...   
2025       0.25      156750.0   
           0.50      156750.0   
           0.75      156750.0   
           0.90      156750.0   
           0.95      156750.0   

                                                                 fwixd  \
                                                           NGFS CurPol   
birth year quantile                                                      
1950       0.05      13.96349793282630513.96349793282630513.9634979...   
           0.10      14.10893450024848814.10893450024848814.1089345...   
           0.25      15.73859754052452215.73859754052452215.7385975...   
           0.50      19.2035850203540919.2035850203540919.203585020...   
           0.75      21.79755094762475321.79755094762475321.7975509...   
...                                                                ...   
2025       0.25      21.89400460487112522.35254685680905222.8312803...   
           0.50      26.56751152609553727.01497156043013327.4026420...   
           0.75      30.46022018665563631.2108457898310931.81277914...   
           0.90      34.2458828622447734.97138063327883435.63398588...   
           0.95      36.7262264161144737.2930044708721537.765976462...   

                                                                        \
                                                    NGFS Net-Zero 2050   
birth year quantile                                                      
1950       0.05      13.96349793282630513.96349793282630513.9634979...   
           0.10      14.10893450024848814.10893450024848814.1089345...   
           0.25      15.73859754052452215.73859754052452215.7385975...   
           0.50      19.2035850203540919.2035850203540919.203585020...   
           0.75      21.79755094762475321.79755094762475321.7975509...   
...                                                                ...   
2025       0.25      21.89056137254367622.34115814167575622.8247621...   
           0.50      26.5625925224078427.0211768114274327.439697274...   
           0.75      30.50243684240705731.25204121861210631.8871337...   
           0.90      34.28519036110261535.073673008309935.724070010...   
           0.95      36.7681911088450737.38291614546045637.86077962...   

                                                            HI-caution  \
                                                           NGFS CurPol   
birth year quantile                                                      
1950       0.05      69.6600968399944169.6600968399944169.660096839...   
           0.10      71.977255961355671.977255961355671.97725596135...   
           0.25      73.0059524145291373.0059524145291373.005952414...   
           0.50      74.2487586276898874.2487586276898874.248758627...   
           0.75      75.1426904611334575.1426904611334575.142690461...   
...                                                                ...   
2025       0.25      78.3140377631604878.8576985946568879.370704633...   
           0.50      81.6653811810960882.224675523594682.8130503936...   
           0.75      84.5279478421237585.0439201922681585.598270832...   
           0.90      87.8316595064798588.7726514855774389.672163258...   
           0.95      90.4085023791006891.0288207246790691.496805014...   

                                                                        
                                                    NGFS Net-Zero 2050  
birth year quantile                                                     
1950       0.05      69.6600968399944169.6600968399944169.660096839...  
           0.10      71.977255961355671.977255961355671.97725596135...  
           0.25      73.0059524145291373.0059524145291373.005952414...  
           0.

In [ ]:
# PLOT results from RIME-X



## Lifetime exposure 




In [7]:
# Call function that processes results from RIME-X

In [8]:
# Plot results for lifetime exposure using RIME-X results



## Compute multi-model mean and model spread


## Plot results for the multi-model mean and the model spread